### **Content License Agreement**

<font color='red'><b>**WARNING**</b></font> : 본 자료는 업스테이지의 콘텐츠 자산으로, 어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다.

# 실습 — Trace 분석: Failure Taxonomy & Regression 환류
**Production-Grade Agent Engineering · Colab 실습 (Upstage Solar)**

> 목표: 실패한 에이전트 **Trace를 Failure Taxonomy(Prompt·Tool·Retrieval·Workflow)로 분류**하고,
> 그 실패를 **회귀 데이터셋(regression set)에 환류**해 에이전트를 한 사이클 진화시킵니다.

## 0. 셋업
Colab 좌측 🔑 **Secrets**에 `UPSTAGE_API_KEY`를 등록한 뒤, 아래 셀을 실행합니다.

In [13]:
!pip -q install openai

In [14]:
# Upstage Solar 셋업 — Colab 좌측 🔑 Secrets에 'UPSTAGE_API_KEY'를 등록한 뒤 실행합니다.
from google.colab import userdata
UPSTAGE_API_KEY = userdata.get("UPSTAGE_API_KEY")
from openai import OpenAI
client = OpenAI(api_key=UPSTAGE_API_KEY, base_url="https://api.upstage.ai/v1")
MODEL="solar-pro3"
print("Solar 준비 완료:",MODEL)

Solar 준비 완료: solar-pro3


## 1. Failure Taxonomy 정의
실패도 **분류**하면 개선 지점이 보입니다. 4대 카테고리로 시작합니다.

In [15]:
TAXONOMY = {
  "PromptFail":   "지시 모호/컨텍스트 부족/지시-실행 불일치",
  "ToolFail":     "잘못된 도구 선택·파라미터·실행 오류",
  "RetrievalFail":"관련 없는 문서·검색 누락·오래된 정보",
  "WorkflowFail": "무한 루프·비용 폭주·의존성/순서 오류",
}
for k,v in TAXONOMY.items(): print(f"  {k}: {v}")

  PromptFail: 지시 모호/컨텍스트 부족/지시-실행 불일치
  ToolFail: 잘못된 도구 선택·파라미터·실행 오류
  RetrievalFail: 관련 없는 문서·검색 누락·오래된 정보
  WorkflowFail: 무한 루프·비용 폭주·의존성/순서 오류


## 2. 실패 Trace 분류 (규칙 + Solar)
운영에서 모은 실패 Trace들을 분류합니다. 명확한 신호는 규칙으로 처리하고, 애매하면 Solar에게 맡깁니다.

In [21]:
import json
failure_traces = [
  {"id":"F1","trace":"search_news(keyword='') → [] → '결과 없음' 답변", "note":"빈 키워드로 호출"},
  {"id":"F2","trace":"질문 '환불 처리해줘' → 어떤 주문인지 안 물어보고 임의 환불", "note":"컨텍스트 부족"},
  {"id":"F3","trace":"agentA→agentB→agentA→... 6회 위임 후 중단", "note":"위임 루프"},
  {"id":"F4","trace":"RAG가 2023년 문서 검색 → 2026년 질문에 옛 정보로 답", "note":"오래된 검색"},
]

def classify(tr):
    prompt=(f"실패 trace를 아래 카테고리 중 하나로 분류하라.\n"
            + "\n".join(f"- {k}: {v}" for k,v in TAXONOMY.items())
            + f"\n\n[trace] {tr['trace']}\n[note] {tr['note']}\n"
            + '결과는 JSON만: {"category":"...","why":"한 줄 근거"}')
    r=client.chat.completions.create(model=MODEL,
        messages=[{"role":"system","content":"에이전트 실패 분석가. JSON만."},{"role":"user","content":prompt}],
        response_format={"type":"json_object"},temperature=0)
    return json.loads(r.choices[0].message.content)

classified=[]
for tr in failure_traces:
    res=classify(tr); classified.append({**tr,**res})
    print(f"  {tr['id']} → {res.get('category')} : {res.get('why','')[:40]}")

  F1 → PromptFail : 빈 키워드로 호출되어 검색 대상이 없어 결과가 없음
  F2 → PromptFail : 질문에 주문 정보가 명시되지 않아 컨텍스트 부족으로 임의 환불을 시도함
  F3 → WorkflowFail : 위임 루프로 인해 무한 반복이 발생하여 작업이 중단됨
  F4 → RetrievalFail : 검색된 문서가 2023년 자료로, 2026년 질문에 대한 최신 정보가 반


## 3. 회귀 데이터셋 환류
분류된 실패를 **회귀셋**에 추가합니다. 다음 평가(이전 강의들)에서 이 케이스들이 자동으로 다시 검증되어 **재발을 막습니다.**

In [23]:
regression=[]
for f in classified:
    regression.append({"id":f"reg_{f['id']}","source":"production_failure",
                       "category":f["category"],"scenario":f["trace"],
                       "expected":"이 실패가 재발하지 않아야 함"})
with open("regression_set.jsonl","w",encoding="utf-8") as fp:
    for r in regression: fp.write(json.dumps(r,ensure_ascii=False)+"\n")
print(f"회귀셋 저장: regression_set.jsonl ({len(regression)}건)")
from collections import Counter
print("실패 분포:", dict(Counter(f["category"] for f in classified)))

print(json.dumps(r,ensure_ascii=False)+"\n")

회귀셋 저장: regression_set.jsonl (4건)
실패 분포: {'PromptFail': 2, 'WorkflowFail': 1, 'RetrievalFail': 1}
{"id": "reg_F4", "source": "production_failure", "category": "RetrievalFail", "scenario": "RAG가 2023년 문서 검색 → 2026년 질문에 옛 정보로 답", "expected": "이 실패가 재발하지 않아야 함"}



## 4. 환류 루프 (Continuous Improvement)
```
프로덕션 트래픽 → (1~5% 샘플) → 온라인 평가 → 실패 감지
   → Failure Taxonomy 분류 → 회귀셋에 추가 → 근본원인별 개선(프롬프트/도구/RAG/흐름)
   → 회귀 테스트 통과 확인 → 배포 (그리고 반복)
```
가장 많은 실패 카테고리부터 고치면 ROI가 가장 큽니다. 이것이 **자기 진화하는 에이전트**의 엔진입니다.

In [24]:
top = Counter(f["category"] for f in classified).most_common(1)[0]
print(f"우선 개선 대상: {top[0]} ({top[1]}건) — 근본원인을 고치고 회귀셋으로 재검증")

우선 개선 대상: PromptFail (2건) — 근본원인을 고치고 회귀셋으로 재검증


## 5. Hermes 스타일 자기진화 — 실패에서 `SKILL.md` 작성·재사용

실제 오픈소스 에이전트 **Hermes Agent**(Nous Research, MIT)는 반복 실패/패턴을 인식하면 스스로
**`SKILL.md`**(YAML frontmatter + 지침)를 작성하고, 사용 중 다듬어 재사용합니다. 앞서 본
*Reflexion·Self-refine* 의 실제 제품 형태입니다. 여기선 그 핵심을 **미니로 재현** — 위에서 찾은
**최다 실패 유형**으로 `SKILL.md`를 자동 작성하고, 다음 처리에 **주입**해 같은 실패를 막는지 확인합니다.


In [25]:
# (5-1) 최다 실패 유형 → 재사용 스킬(SKILL.md) 자동 작성 (Hermes의 자율 스킬 생성 미니)
top_cat = top[0]
examples = [f["trace"] for f in classified if f["category"] == top_cat]

skill_prompt = (
    f"너는 에이전트 자기개선 모듈이다. 반복 실패 유형 '{top_cat}'을 예방하는 재사용 스킬을 작성하라.\n"
    f"실패 예시: {examples}\n\n"
    "출력은 SKILL.md 형식 마크다운만:\n"
    "---\nname: <짧은-스킬이름>\ntrigger: <적용 상황>\ndescription: <한 줄>\n---\n"
    "그 아래 '행동 전 체크리스트' 3~5개(불릿)."
)
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role":"system","content":"실패 예방 스킬 작성기. SKILL.md 형식 마크다운만."},
              {"role":"user","content":skill_prompt}],
    temperature=0.2)
skill_md = r.choices[0].message.content
with open("SKILL.md","w",encoding="utf-8") as fp:
    fp.write(skill_md)
print(f"[{top_cat}] 예방 SKILL.md 생성\n"); print(skill_md)


[PromptFail] 예방 SKILL.md 생성

---
name: PromptFailPrevention
trigger: 사용자가 모호한 프롬프트를 제공하거나 빈 입력을 전달했을 때
description: 빈/불완전한 프롬프트를 감지하고, 필요한 추가 정보를 요청하여 실패를 방지한다.
---
- **프롬프트 길이 확인**: 입력 문자열이 비어 있거나 최소 길이(예: 3자) 미만인지 검사한다.  
- **필수 키워드 존재 여부**: 핵심 키워드(예: `keyword`, `order_id`, `refund`)가 포함되었는지 확인한다.  
- **의도 파악**: 사용자가 원하는 작업(검색, 환불, 예약 등)을 명확히 추론할 수 있는지 판단한다.  
- **추가 질문 준비**: 부족한 정보가 있을 경우, 구체적인 질문을 미리 정의해 두고 사용자에게 요청한다.  
- **응답 템플릿 선택**: 정보가 충분하면 정상 처리, 부족하면 “추가 정보가 필요합니다.”와 같은 표준 응답을 사용한다.


In [26]:
# (5-2) 생성한 스킬을 주입해 동일 입력 재처리 — 스킬 재사용으로 자가개선 확인
def handle(task, skill=None):
    sys = "너는 신중한 에이전트다. 불확실하면 행동 전에 질문하거나 검증한다."
    if skill:
        sys += "\n\n[적용된 SKILL.md]\n" + skill
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role":"system","content":sys},{"role":"user","content":task}],
        temperature=0)
    return r.choices[0].message.content

risky = "주문 환불 처리해줘"   # 예전엔 컨텍스트 부족으로 임의 환불(F2)하던 입력
print("● 스킬 없이:\n", handle(risky)[:300])
print("\n● SKILL.md 적용 후:\n", handle(risky, skill_md)[:300])
print("\n기대: 스킬 적용 시 '어떤 주문인지' 먼저 확인 → 같은 실패를 스스로 예방(자기진화 1사이클).")


● 스킬 없이:
 안녕하세요. 주문 환불을 도와드리기 위해서는 몇 가지 정보가 필요합니다. 아래 항목들을 알려주시면 정확한 환불 처리를 진행할 수 있습니다.

1. **주문 번호** (예: 123456789)  
2. **환불을 원하시는 결제 수단** (예: 신용카드, 계좌이체, 페이팔 등)  
3. **환불 사유** (예: 상품 불량, 주문 착오, 단순 변심 등)  
4. **환불 받으실 계좌 정보** (은행명, 계좌번호, 예금주명) – 계좌이체를 선택하신 경우  
5. **환불을 원하시는 금액** (전체 환불인지, 부분 환불인지)  

위 정보를

● SKILL.md 적용 후:
 주문 환불을 처리해 드리기 위해서는 몇 가지 정보가 필요합니다. 아래 항목들을 알려주시면 바로 진행해 드릴 수 있습니다.

1. **주문 번호 (order_id)** – 환불을 요청할 주문의 고유 번호  
2. **환불 사유** – 예: 상품 불량, 사이즈/색상 오류, 단순 변심 등  
3. **환불받을 계좌 정보** – 은행명, 계좌번호, 예금주명 (필요 시)  
4. **환불 방법** – 원래 결제 수단(카드, 계좌)으로 환불할지, 다른 방법으로 할지  

위 정보를 알려주시면 바로 환불 절차를 시작하겠습니다. 추가로 궁금한 점

기대: 스킬 적용 시 '어떤 주문인지' 먼저 확인 → 같은 실패를 스스로 예방(자기진화 1사이클).


## 직접 해보기
새로운 실패 사례를 분류 파이프라인에 넣어 보세요. `failure_traces`에 실패 1건을 추가하고, 분류 결과와 회귀셋 반영을 확인합니다.

In [9]:
# TODO: 1) failure_traces 에 새 실패 trace 1건(id/trace/note)을 추가하고
#       2) classify() 로 분류한 뒤
#       3) regression_set.jsonl 에 반영되는지 확인하세요.


In [10]:
# ▼ 정답 예시 — 직접 해본 뒤 주석을 풀고 실행하세요
# new_tr = {"id": "F5", "trace": "get_detail('없는 헤드라인') → '상세 정보 없음' → 그대로 단정 답변",
#           "note": "빈 검색 결과를 검증 없이 사용"}
# res = classify(new_tr)
# print("F5 →", res.get("category"), ":", res.get("why", ""))


## 정리
- 실패는 **Failure Taxonomy**로 분류해야 개선 지점이 보입니다.
- 분류된 실패를 **회귀셋에 환류**하면 같은 실패가 재발하지 않습니다(이전 강의들 평가와 연결됩니다).
- 이 **수집→분류→환류→재배포** 루프가 에이전트를 스스로 진화시킵니다.

---
## (부록) 실제 Hermes Agent — 실행 스크립트 & 실행 방법

> 위 미니 구현의 '진짜 버전'. Hermes는 **대화형 TUI**라 로컬 터미널 / WSL2 / macOS에서 실행하세요(Colab 셀은 비대화형이라 부적합).

**1) 설치** (Linux·macOS·WSL2 — Python 3.11 + uv 기반)
```bash
curl -fsSL https://hermes-agent.nousresearch.com/install.sh | bash
source ~/.bashrc        # 또는 ~/.zshrc
```

**2) provider 설정 → 실행**
```bash
hermes setup            # provider/모델 설정 (hermes model 로 모델만 변경 가능)
hermes                  # 대화 시작 (TUI)
```
```

출처: 공식 docs <https://hermes-agent.nousresearch.com/docs/> · GitHub <https://github.com/nousresearch/hermes-agent>
